Code to Extract data from vloume and convert dataframe into table

In [0]:
ball_by_ball_df = spark.read.csv("/Volumes/workspace/karan/ipl/Ball_By_Ball.csv/", header=True, inferSchema=True)
match_df = spark.read.csv("/Volumes/workspace/karan/ipl/Match.csv", header=True, inferSchema=True)
player_df = spark.read.csv("/Volumes/workspace/karan/ipl/Player.csv/", header=True, inferSchema=True)
team_df = spark.read.csv("/Volumes/workspace/karan/ipl/Team.csv", header=True, inferSchema=True)
player_match_df = spark.read.csv("/Volumes/workspace/karan/ipl/Player_match.csv", header=True, inferSchema=True)

ball_by_ball_df.createOrReplaceTempView("Ball_By_Ball")
match_df.createOrReplaceTempView("Match")
player_df.createOrReplaceTempView("Player")
team_df.createOrReplaceTempView("Team")
player_match_df.createOrReplaceTempView("Player_match")

most expensive over

In [0]:
query = """
SELECT 
  match_id,
  innings_no,
  over_id,
  Bowler,b.Team_Batting,b.Team_Bowling,
  p.player_name AS bowler_name,
  SUM(runs_scored + extra_runs) AS total_runs
FROM Ball_By_Ball b join player p 
on b.Bowler = p.Player_Id

GROUP BY match_id, innings_no, over_id,Bowler,p.player_name,b.Team_Batting,b.Team_Bowling
ORDER BY total_runs desc

"""

expensive_overs_df = spark.sql(query)
expensive_overs_df.createOrReplaceTempView("Expensive_Overs")
display(expensive_overs_df)

Most number of wicket by each bowler based on match

In [0]:
%sql
select t.Bowler,sum(t.total),t.player_name,t.total_match from (select Bowler, Sum(LBW+Bowled+Caught)as total,p.player_name,count(match_id) as total_match from ball_by_ball b join Player p
on b.Bowler=p.Player_Id
group by b.Bowler,p.player_name,MatcH_id
order by Sum(LBW+Bowled+Caught) desc,p.player_name ) 
as t
group by t.player_name,t.bowler,t.total_match
order by sum(t.total) desc


to find the 3th highest wicket taker from each team

In [0]:
%sql
select * from(select e.*,row_number() over(partition by Team_Bowling order by total_wicket desc) as rnk   from (select bowler,Team_Bowling,sum(Bowler_Wicket) as total_wicket from ball_by_ball
group by Bowler,Team_Bowling
order by Team_Bowling,sum(Bowler_Wicket)desc,Bowler) as e) X
where X.RNK=3



get the 3th highest wicket taker from each team

In [0]:
%sql
select * from(select e.*,dense_rank() over(partition by Team_Bowling order by total_wicket desc) as rnk   from (select bowler,Team_Bowling,sum(Bowler_Wicket) as total_wicket from ball_by_ball
group by Bowler,Team_Bowling
order by Team_Bowling,sum(Bowler_Wicket)desc,Bowler) as e) X


Use of lag and lead function

In [0]:
%sql
select X.*,
CASE WHEN TOTAL_WICKET>lag(TOTAL_WICKET) over(partition by Team_Bowling order by BOWLER) THEN 'GOOD BOWLER'
WHEN TOTAL_WICKET<LAG(TOTAL_WICKET) over(partition by Team_Bowling order by BOWLER) THEN 'AVG BOWLER'
WHEN TOTAL_WICKET=lag(TOTAL_WICKET) over(partition by Team_Bowling order by BOWLER) THEN 'SAME_SPRIT'
END STATUS_OF_BOWLER 
 from(select e.*,dense_rank() over(partition by Team_Bowling order by total_wicket desc) as rnk   from (select bowler,Team_Bowling,sum(Bowler_Wicket) as total_wicket from ball_by_ball
group by Bowler,Team_Bowling
order by Team_Bowling,sum(Bowler_Wicket)desc,Bowler) as e) X

Use of group by to find most no of player from one team

In [0]:
%sql
select country_name,count(Player_Id) as total_player from player
group by country_name
order by total_player;

Highest run getter from each Team


In [0]:
%sql
Select X.*,dense_rank() over(partition by Team_Batting order by total_Run desc) as Rank   from (select Team_Batting, Striker,sum(Runs_Scored) total_Run,p.player_name from ball_by_ball b join Player p
on b.Striker=p.Player_Id
group by Striker,player_name,Team_Batting 
order by total_run desc) X;



dense rank

In [0]:
%sql

Select X.*,dense_rank() over(partition by Team_Batting order by total_Run desc) as Rank   from (select Match_id,Team_Batting, Striker,sum(Runs_Scored) total_Run,p.player_name from ball_by_ball b join Player p
on b.Striker=p.Player_Id
group by Striker,player_name,Team_Batting,Match_id 
order by total_run desc) X;

maximum run scored in an inning by player

In [0]:
%sql
select Match_id,Team_Batting, Striker,sum(Runs_Scored) total_Run,p.player_name from ball_by_ball b join Player p
on b.Striker=p.Player_Id
group by Striker,player_name,Team_Batting,Match_id 
order by total_run desc, b.match_id

to get the maximum balled over and on that over maximum run

In [0]:
%sql
select MatcH_id,Over_id,Innings_No,sum(total) from 
(select MatcH_id,Over_id,Innings_No,Runs_Scored+Extra_runs as total from Ball_By_Ball 
where (MatcH_id,Over_id) in
(select MatcH_id,Over_id from
(select match_id,over_id,bowler,p.player_name, max(ball_id )from ball_by_ball b join player p on b.Bowler=p.Player_Id
group by MatcH_id,Over_id,Bowler,p.player_name
order by max(ball_id) desc)))
group by MatcH_id,Innings_No,Over_id
order by sum(total) desc




over bowled by more then one bowler

In [0]:
%sql
 select MatcH_id,Innings_No,Over_id from ball_by_ball
group by MatcH_id,Innings_No,over_id
having count(distinct bowler)>1
order by MatcH_id,Innings_No,Over_id

total wicket by each bowler

In [0]:
%sql
select bowler ,sum(Bowler_Wicket) total_wicket,p.Player_Name from ball_by_ball join player p on ball_by_ball.Bowler=p.Player_Id
group by Bowler,p.Player_Name
order by sum(Bowler_Wicket) desc

### Write mode 
append


In [0]:
# Check who you are
spark.sql("SELECT current_user()").show()

# Check grants on the volume
spark.sql("SHOW GRANTS ON VOLUME workspace.default.karan").show()

In [0]:
# Grant all privileges to a specific user
spark.sql("GRANT ALL PRIVILEGES ON VOLUME workspace.default.karan TO `karan8726534242@gmail.com`")

# Specific write permission only
spark.sql("GRANT WRITE_VOLUME ON VOLUME workspace.default.karan TO `karan8726534242@gmail.com`")

In [0]:
df=spark.sql("""select bowler ,sum(Bowler_Wicket) total_wicket,p.Player_Name from ball_by_ball join player p on ball_by_ball.Bowler=p.Player_Id
group by Bowler,p.Player_Name
order by sum(Bowler_Wicket) desc""")

# Ensure the output directory exists and you have write permissions.
# For /Volumes, make sure the workspace and volume are mounted and accessible.

df.write.mode("append").option("header", "true").csv("dbfs:/Volumes/workspace/default/karan/karan.csv")
df.write.mode("append").parquet("dbfs:/Volumes/workspace/default/karan/karan.parquet")
df.write.mode("append").json("dbfs:/Volumes/workspace/default/karan/karan.json")
df.write.mode("append").orc("dbfs:/Volumes/workspace/default/karan/karan.orc")


In [0]:
base_path = "dbfs:/Volumes/workspace/default/karan"

# Step 1: Delete the old karan.csv folder that was created previously
dbutils.fs.rm(f"{base_path}/karan.csv", recurse=True)
dbutils.fs.rm(f"{base_path}/karan.json", recurse=True)
dbutils.fs.rm(f"{base_path}/karan.orc", recurse=True)
dbutils.fs.rm(f"{base_path}/karan.parquet", recurse=True)

# Also clean old format folders from previous runs
dbutils.fs.rm(f"{base_path}/csv", recurse=True)
dbutils.fs.rm(f"{base_path}/json", recurse=True)
dbutils.fs.rm(f"{base_path}/orc", recurse=True)
dbutils.fs.rm(f"{base_path}/parquet", recurse=True)
dbutils.fs.rm(f"{base_path}/delta", recurse=True)

print("✅ Cleanup done!")

In [0]:
df = spark.sql("""
    select bowler, sum(Bowler_Wicket) total_wicket, p.Player_Name 
    from ball_by_ball 
    join player p on ball_by_ball.Bowler = p.Player_Id
    group by Bowler, p.Player_Name
    order by sum(Bowler_Wicket) desc
""")

base_path = "dbfs:/Volumes/workspace/default/karan"

def save_with_custom_name(df, temp_path, final_path, fmt, options={}):
    w = df.coalesce(1).write.mode("overwrite")
    for k, v in options.items():
        w = w.option(k, v)
    getattr(w, fmt)(temp_path)

    files = dbutils.fs.ls(temp_path)
    part_file = [f.path for f in files if f.name.startswith("part-")][0]
    dbutils.fs.mv(part_file, final_path)
    dbutils.fs.rm(temp_path, recurse=True)
    print(f"✅ Saved: {final_path}")

save_with_custom_name(df, f"{base_path}/temp_csv",     f"{base_path}/karan.csv",     "csv",     {"header": "true"})
save_with_custom_name(df, f"{base_path}/temp_parquet", f"{base_path}/karan.parquet", "parquet")
save_with_custom_name(df, f"{base_path}/temp_json",    f"{base_path}/karan.json",    "json")
save_with_custom_name(df, f"{base_path}/temp_orc",     f"{base_path}/karan.orc",     "orc")


In [0]:
df = spark.sql("""
    select bowler, sum(Bowler_Wicket) total_wicket, p.Player_Name 
    from ball_by_ball 
    join player p on ball_by_ball.Bowler = p.Player_Id
    group by Bowler, p.Player_Name
    order by sum(Bowler_Wicket) desc
""")

base_path = "dbfs:/Volumes/workspace/default/karan"

# ── Helper Function ─────────────────────────────────────────────────────────
def save_with_custom_name(df, temp_path, final_path, fmt, mode, options={}):
    w = df.coalesce(1).write.mode(mode)
    for k, v in options.items():
        w = w.option(k, v)
    getattr(w, fmt)(temp_path)

    files = dbutils.fs.ls(temp_path)
    part_file = [f.path for f in files if f.name.startswith("part-")][0]
    dbutils.fs.mv(part_file, final_path)
    dbutils.fs.rm(temp_path, recurse=True)
    print(f"✅ Saved: {final_path}")


# ════════════════════════════════════════════════════════════════════════════
# MODE 1: OVERWRITE
# - Deletes existing data and writes fresh data
# - Use when you want latest data only, no duplicates
# - Folder: write_mode_overwrite/
# ════════════════════════════════════════════════════════════════════════════
print("\n📁 MODE: OVERWRITE — Deletes old data, writes new data fresh")

overwrite_path = f"{base_path}/write_mode_overwrite"

save_with_custom_name(df, f"{overwrite_path}/temp_csv",     f"{overwrite_path}/karan.csv",     "csv",     "overwrite", {"header": "true"})
save_with_custom_name(df, f"{overwrite_path}/temp_parquet", f"{overwrite_path}/karan.parquet", "parquet", "overwrite")
save_with_custom_name(df, f"{overwrite_path}/temp_json",    f"{overwrite_path}/karan.json",    "json",    "overwrite")
save_with_custom_name(df, f"{overwrite_path}/temp_orc",     f"{overwrite_path}/karan.orc",     "orc",     "overwrite")


# ════════════════════════════════════════════════════════════════════════════
# MODE 2: APPEND
# - Adds new data to existing data without touching old records
# - Use for incremental loads, logs, event data
# - Folder: write_mode_append/
# ════════════════════════════════════════════════════════════════════════════
print("\n📁 MODE: APPEND — Adds new data on top of existing data")

append_path = f"{base_path}/write_mode_append"

save_with_custom_name(df, f"{append_path}/temp_csv",     f"{append_path}/karan.csv",     "csv",     "append", {"header": "true"})
save_with_custom_name(df, f"{append_path}/temp_parquet", f"{append_path}/karan.parquet", "parquet", "append")
save_with_custom_name(df, f"{append_path}/temp_json",    f"{append_path}/karan.json",    "json",    "append")
save_with_custom_name(df, f"{append_path}/temp_orc",     f"{append_path}/karan.orc",     "orc",     "append")


# ════════════════════════════════════════════════════════════════════════════
# MODE 3: IGNORE
# - Does nothing if file already exists (silent skip)
# - Use when you don't want to overwrite existing data accidentally
# - Folder: write_mode_ignore/
# ════════════════════════════════════════════════════════════════════════════
print("\n📁 MODE: IGNORE — Skips writing if file already exists")

ignore_path = f"{base_path}/write_mode_ignore"

save_with_custom_name(df, f"{ignore_path}/temp_csv",     f"{ignore_path}/karan.csv",     "csv",     "ignore", {"header": "true"})
save_with_custom_name(df, f"{ignore_path}/temp_parquet", f"{ignore_path}/karan.parquet", "parquet", "ignore")
save_with_custom_name(df, f"{ignore_path}/temp_json",    f"{ignore_path}/karan.json",    "json",    "ignore")
save_with_custom_name(df, f"{ignore_path}/temp_orc",     f"{ignore_path}/karan.orc",     "orc",     "ignore")


# ════════════════════════════════════════════════════════════════════════════
# MODE 4: ERROR IF EXISTS (default)
# - Throws an error if the destination already has data
# - Safest mode — prevents accidental overwrites
# - Folder: write_mode_errorifexists/
# ════════════════════════════════════════════════════════════════════════════
print("\n📁 MODE: ERRORIFEXISTS — Throws error if file already exists")

error_path = f"{base_path}/write_mode_errorifexists"

save_with_custom_name(df, f"{error_path}/temp_csv",     f"{error_path}/karan.csv",     "csv",     "errorIfExists", {"header": "true"})
save_with_custom_name(df, f"{error_path}/temp_parquet", f"{error_path}/karan.parquet", "parquet", "errorIfExists")
save_with_custom_name(df, f"{error_path}/temp_json",    f"{error_path}/karan.json",    "json",    "errorIfExists")
save_with_custom_name(df, f"{error_path}/temp_orc",     f"{error_path}/karan.orc",     "orc",     "errorIfExists")


print("\n🎉 All write modes completed!")


Window funcion

In [0]:
from pyspark.sql import Window
from pyspark.sql.functions import row_number, rank, dense_rank

# Create DataFrame for employee table with 10 rows
data = [
    (1, "Alice", "HR", 5000),
    (2, "Bob", "Finance", 6000),
    (3, "Charlie", "HR", 5500),
    (4, "David", "IT", 7000),
    (5, "Eve", "Finance", 6200),
    (6, "Frank", "IT", 7100),
    (7, "Grace", "HR", 5200),
    (8, "Heidi", "Finance", 6300),
    (9, "Ivan", "IT", 6900),
    (10, "Judy", "HR", 5400)
]
columns = ["emp_id", "name", "department", "salary"]
df = spark.createDataFrame(data, columns)

window = Window.partitionBy("department").orderBy(df.salary.desc())

df_windowed = df.withColumn("row_number", row_number().over(window)) \
    .withColumn("rank", rank().over(window)) \
    .withColumn("dense_rank", dense_rank().over(window))

display(df_windowed)

head to head match between teams

In [0]:
%sql
select 
  least(Team_Batting, Team_Bowling) as Team1,
  greatest(Team_Batting, Team_Bowling) as Team2,
  count(distinct MatcH_id) as total_head_to_head_matches
from ball_by_ball
where Team_Batting != Team_Bowling
group by least(Team_Batting, Team_Bowling), greatest(Team_Batting, Team_Bowling)
order by total_head_to_head_matches desc,Team1, Team2

maximum  no of wides in a match

In [0]:
%sql
select MatcH_id,Innings_No,sum(Wides) from ball_by_ball
group by MatcH_id,Innings_No
order by sum(Wides) desc;

maximum no of sixes in match

In [0]:
%sql
select b.MatcH_id,b.Innings_No,count(Runs_Scored),m.Team1,m.Team2 from ball_by_ball b join match m
on b.MatcH_id=m.MatcH_id
where Runs_Scored=6
group by b.MatcH_id,b.Innings_No,m.Team1,m.Team2
order by count(Runs_Scored)desc

Player_sts based on run scored

In [0]:
%sql
select Striker,p.player_name,sum(Runs_Scored),count(Runs_Scored) from ball_by_ball b join player p 
on b.Striker=p.player_id
where Runs_Scored=3
group by Striker,p.player_name
order by sum(Runs_Scored) desc

win or lost based on match

In [0]:
%sql
with team_runs as (
  select 
    MatcH_id,
    Team_Batting,
    Team_Bowling,
    sum(Runs_Scored) as total_runs
  from ball_by_ball
  group by MatcH_id, Team_Batting, Team_Bowling
)
select 
  tr1.MatcH_id,
  tr1.Team_Batting,
  tr1.Team_Bowling,
  tr1.total_runs,
  case 
    when tr1.total_runs > tr2.total_runs then 'win'
    else 'loss'
  end as result
from team_runs tr1
left join team_runs tr2
  on tr1.MatcH_id = tr2.MatcH_id
  and tr1.Team_Batting = tr2.Team_Bowling
  and tr1.Team_Bowling = tr2.Team_Batting
order by tr1.MatcH_id, tr1.Team_Batting, tr1.Team_Bowling;

In [0]:
# Calculate total runs for each team in each match
team_runs_df = spark.sql("""
    select 
        MatcH_id,
        Team_Batting,
        Team_Bowling,
        sum(Runs_Scored) as total_runs
    from ball_by_ball
    group by MatcH_id, Team_Batting, Team_Bowling
""")

# Pivot to get both teams' runs in one row
pivot_df = team_runs_df.groupBy("MatcH_id") \
    .pivot("Team_Batting") \
    .agg({"total_runs": "max"})

# Join original team_runs_df with pivoted runs to compare
result_df = team_runs_df.withColumn(
    "opponent_runs",
    team_runs_df["Team_Bowling"].cast("string")
).join(
    pivot_df,
    team_runs_df["MatcH_id"] == pivot_df["MatcH_id"],
    "left"
)

# Determine win/loss
result_df = result_df.withColumn(
    "result",
    (result_df["total_runs"] > result_df[result_df["opponent_runs"]]).when("win").otherwise("loss")
)

display(result_df.select("MatcH_id", "Team_Batting", "Team_Bowling", "total_runs", "result"))

longest over in particular match

In [0]:
%sql
select match_id,over_id,innings_no,count(ball_id) from ball_by_ball
group by match_id,Over_id,Innings_No
order by count(Ball_id) desc;

maximum no of wicket in inning

In [0]:
%sql
select t.Innings_No,t.total_wicket,t.bowler from (select MatcH_id,Innings_No,bowler,sum(Bowler_Wicket) as total_wicket from ball_by_ball
group by MatcH_id,Innings_No,bowler
order by sum(Bowler_Wicket) desc  ) t
group by t.bowler,t.total_wicket,t.innings_no
order by t.Innings_No,t.total_wicket desc,t.bowler


maximum no of sixes or fours in an over

In [0]:
%sql
select match_id,over_id,innings_no,Runs_Scored,count(Runs_Scored) from ball_by_ball
where Runs_Scored=4
group by match_id,Over_id,Innings_No,Runs_Scored
order by count(Runs_Scored) desc,MatcH_id,Over_id,Innings_No

sixes and fours in each session

In [0]:
%sql
select 
  Season,
  sum(case when Runs_Scored = 6 then 1 else 0 end) as total_sixes,
  sum(case when Runs_Scored = 4 then 1 else 0 end) as total_fours,
  sum(Wides) as total_wides,
  sum(Bowler_Wicket) as total_wickets
from ball_by_ball
group by Season
order by Season

total wic by each season

In [0]:
%sql
select Team_Bowling,Bowler,sum(Bowler_Wicket),Season from ball_by_ball
group by Season,Team_Bowling,Bowler
order by Team_Bowling,Season,sum(Bowler_Wicket) desc;


wicket taker by each team for each season

In [0]:
%sql
select Team_Bowling, Bowler, sum(Bowler_Wicket), season, dense_rank() over (partition by Team_Bowling, Season order by sum(Bowler_Wicket) desc) as dense_rank_wicket
from ball_by_ball
group by Team_Bowling, bowler, season
order by Team_Bowling, Season


most number of wicket in an over

In [0]:
%sql
select b.match_id, Innings_No, Over_id,(sum(Bowler_Wicket)+Run_out) as total_wicket,b.bowler,m.team1,m.Team2,p.Player_Name from ball_by_ball b join match m on
b.MatcH_id=m.MatcH_id
join Player p on b.Bowler=p.player_id
group by b.match_id, Innings_No, Over_id,Run_out,bowler,m.team1,m.Team2,p.Player_Name
order by total_wicket desc

query to find no of bowled greate then or equal to 2 in a particular match more then one time

In [0]:
%sql
select 
  match_id
from (
  select 
    MatcH_id as match_id,
    Innings_No,
    Over_id,
    sum(bowled) as total_bowled
  from ball_by_ball
  group by MatcH_id, Innings_No, Over_id
) t
where total_bowled >= 2
group by match_id
having count(*) >= 2
order by match_id

In [0]:
%sql
SELECT MatcH_id, Innings_No, Striker, COUNT(CASE WHEN Runs_Scored = 0 THEN 1 END) AS dotball, COUNT(*) AS total_ball_played FROM ball_by_ball GROUP BY MatcH_id, Innings_No, Striker ORDER BY dotball DESC;

longest partnership in match

In [0]:
%sql
select * from (
  select MatcH_id,Innings_No,Striker,NONStriker_SK,Season,sum(Runs_Scored) as total,dense_rank() over (partition by season order by sum(Runs_Scored) desc) as rank  
  from ball_by_ball
  group by MatcH_id,Innings_No,Striker,NONStriker_SK,Season
)as m 
where rank <= 3
order by m.season,total desc


In [0]:
%sql
select Season,count(*) as COUNT 
from (
  select MatcH_id, Innings_No, Striker, NONStriker_SK, Season, sum(Runs_Scored) as total
  from ball_by_ball
  group by MatcH_id, Innings_No, Striker, NONStriker_SK, Season
  having sum(Runs_Scored) > 100
)
group by Season
order by COUNT desc, Season

maximum no madin over by bowler by match 

In [0]:
%sql
select match_id,Innings_No,Over_id,Bowler,sum(Runs_Scored)  from ball_by_ball
group by match_id,Innings_No,Over_id,Bowler 
having sum(Runs_Scored)=0
order by Bowler,match_id,Innings_No,Over_id

In [0]:
%sql
select Bowler,count(over_id) mad_over,p.Player_name  from ball_by_ball b join player p on b.bowler=p.player_id
group by Over_id,Bowler,p.player_name
having sum(Runs_Scored)=0
order by mad_over desc

In [0]:
%sql
select MatcH_id,Bowler,sum(Bowled)as total,p.player_name   from ball_by_ball b join player p on
p.player_id=b.bowler
group by Bowler,p.player_name,MatcH_id
order by total desc

Find the top 10 bowlers with the best economy rate in overs 16–20, considering only bowlers with at least 200 balls bowled in those overs

In [0]:
query = """
SELECT 
  b.Bowler,
  p.player_name,
  SUM(b.Runs_Scored) AS total_runs,
  COUNT(*) AS balls_bowled,
  SUM(b.Runs_Scored) / (COUNT(*) / 6.0) AS economy_rate
FROM ball_by_ball b
JOIN player p ON b.Bowler = p.player_id
WHERE b.Over_id BETWEEN 16 AND 20
GROUP BY b.Bowler, p.player_name
HAVING COUNT(*) >= 200
ORDER BY economy_rate ASC
LIMIT 10
"""

df = spark.sql(query)
display(df)

In [0]:
%sql
select Striker,p.Player_Name,sum(Runs_Scored) from ball_by_ball b join player p on
b.striker=p.player_id
where p.Batting_hand='Right-hand bat'
group by Striker,p.Player_Name
order by sum(Runs_Scored) desc



In [0]:
%sql
select ManOfMach,count(ManOfMach) as total  from match
group by ManOfMach
order by total desc;

find out the players who won man of the match but opposition team win the match

In [0]:
df = spark.sql("""
SELECT 
  pm.Player_Id,
  pm.Player_Name,
  pm.Match_Id,
  m.ManOfMach,
  m.match_winner,
  pm.is_manofThematch,
  pm.IsPlayers_Team_won
FROM Player_match pm
JOIN match m ON pm.Match_Id = m.match_id
WHERE pm.is_manofThematch = 1
  AND pm.IsPlayers_Team_won = 0
""")
display(df)

 CELL 5 — Query 4: Subquery — Players who scored above
# the overall IPL average strike rate
# Subquery in WHERE clause
# ============================================================

In [0]:
spark.sql("""
    SELECT
        p.Player_Name,
        ROUND(SUM(b.Runs_Scored) * 100.0 / COUNT(*), 2) AS strike_rate,
        SUM(b.Runs_Scored) AS total_runs
    FROM ball_by_ball b
    JOIN player p ON b.Striker = p.Player_Id
    GROUP BY p.Player_Name
    HAVING strike_rate > (
        -- Subquery: calculates overall IPL average strike rate
        SELECT ROUND(SUM(Runs_Scored) * 100.0 / COUNT(*), 2)
        FROM ball_by_ball
    )
    AND COUNT(*) > 300   -- minimum 300 balls faced for relevance
    ORDER BY strike_rate DESC
    LIMIT 15
""").display()

Maximum runs givien by baller in single over

In [0]:
%sql
select match_id,Innings_No,bowler,sum(Runs_Scored) total from ball_by_ball
group by match_id,Innings_No,Bowler
order by total desc

In [0]:
# Question:
# Find the top 5 batsmen with the highest average runs per match in a single season (minimum 10 matches played in that season), 
# along with their team name and season. Use window functions to rank them within each season, and optimize the query using CTEs.

query = """
WITH player_match_runs AS (
  SELECT
    b.Season,
    b.Striker AS player_id,
    pm.Team_Id AS team_id,
    SUM(b.Runs_Scored) AS total_runs,
    COUNT(DISTINCT b.MatcH_id) AS matches_played
  FROM ball_by_ball b
  JOIN Player_match pm
    ON b.MatcH_id = pm.Match_Id AND b.Striker = pm.Player_Id
  GROUP BY b.Season, b.Striker, pm.Team_Id
),
qualified_players AS (
  SELECT
    *,
    total_runs * 1.0 / matches_played AS avg_runs_per_match
  FROM player_match_runs
  WHERE matches_played >= 10
),
ranked_players AS (
  SELECT
    qp.Season,
    qp.player_id,
    p.Player_Name,
    t.Team_Name,
    qp.avg_runs_per_match,
    ROW_NUMBER() OVER (PARTITION BY qp.Season ORDER BY qp.avg_runs_per_match DESC) AS season_rank
  FROM qualified_players qp
  JOIN player p ON qp.player_id = p.Player_Id
  JOIN Team t ON qp.team_id = t.Team_Id
)
SELECT
  Season,
  player_id,
  Player_Name,
  Team_Name,
  ROUND(avg_runs_per_match, 2) AS avg_runs_per_match,
  season_rank
FROM ranked_players
WHERE season_rank <= 5
ORDER BY Season, season_rank
"""

df = spark.sql(query)
display(df)

In [0]:
%sql
select match_id,innings_no,over_id,sum(Run_out)   from ball_by_ball
group by match_id,innings_no,over_id
order by sum(Run_out) desc

In [0]:
%sql
select Match_Id,Player_Name from player_match
where Role_Desc='Keeper'
order by match_id;

In [0]:
%sql
select rnk.* from (select s.*, row_number() over (partition by team_batting order by total_runs desc)rk  from (select Team_Batting,Striker,sum(Runs_Scored) as total_runs from ball_by_ball
Group by Team_Batting,Striker
order by Team_Batting,total_runs desc) s) rnk
where rnk.rk=3

Q1 — Ground Prediction (Bat First vs Chase)
The logic identifies whether the team batting first won by checking: if toss winner chose to bat AND won, OR if toss winner chose to field AND the other team won (meaning the team that batted first won). Chepauk (Chennai) strongly favors batting first at 63.8%, while Chinnaswamy (Bangalore) and Eden Gardens strongly favor chasing. The threshold milestones are ≥55% = bat first, ≥45% = neutral, else chase.


In [0]:
%sql
-- Q1: Ground-wise Bat-First Win Prediction
SELECT
    Venue_Name, City_Name,
    COUNT(*) AS total_matches,
    SUM(CASE
        WHEN (LOWER(Toss_Name)='bat'   AND match_winner=Toss_Winner)
          OR (LOWER(Toss_Name)='field' AND match_winner!=Toss_Winner
              AND Outcome_Type='Result')
        THEN 1 ELSE 0 END)                        AS bat_first_wins,
    ROUND(bat_first_wins*100.0/COUNT(*),2)         AS bat_first_win_pct,
    CASE
        WHEN bat_first_win_pct >= 55 THEN 'STRONGLY favors Bat First'
        WHEN bat_first_win_pct >= 45 THEN 'NEUTRAL / Toss-Up'
        ELSE                              'STRONGLY favors Chase'
    END                                            AS ground_prediction
FROM match
WHERE Outcome_Type='Result'
GROUP BY Venue_Name, City_Name
HAVING COUNT(*) >= 5
ORDER BY bat_first_win_pct DESC

Q2 — Batsmen Roles (Chase vs 1st Inn)
Uses a pivot on Innings_No (1 or 2) to get strike rate per context. The classification threshold is ±15 strike rate difference — if your chase SR beats your Inn1 SR by 15+, you're a Dominant Chaser. V Sehwag sits as an all-rounder despite his swashbuckling reputation since his SR is consistent across both innings

In [0]:
%sql
-- Q2: Batsmen role — Chaser vs 1st Innings Dominator
WITH batsman_stats AS (
    SELECT b.Striker AS player_sk, p.Player_Name, b.Innings_No,
        SUM(b.Runs_Scored)                        AS total_runs,
        COUNT(*)                                   AS balls_faced,
        ROUND(SUM(b.Runs_Scored)*100.0/COUNT(*),2) AS strike_rate
    FROM ball_by_ball b JOIN player p ON b.Striker=p.PLAYER_SK
    WHERE b.Innings_No IN (1,2)
    GROUP BY b.Striker, p.Player_Name, b.Innings_No
),
pivot AS (
    SELECT Player_Name,
        MAX(CASE WHEN Innings_No=1 THEN total_runs  END) AS inn1_runs,
        MAX(CASE WHEN Innings_No=1 THEN strike_rate END) AS inn1_sr,
        MAX(CASE WHEN Innings_No=2 THEN total_runs  END) AS chase_runs,
        MAX(CASE WHEN Innings_No=2 THEN strike_rate END) AS chase_sr
    FROM batsman_stats GROUP BY Player_Name
)
SELECT *, (COALESCE(inn1_runs,0)+COALESCE(chase_runs,0)) AS total_runs,
    CASE
        WHEN COALESCE(chase_sr,0)-COALESCE(inn1_sr,0) >= 15 THEN 'DOMINANT CHASER'
        WHEN COALESCE(inn1_sr,0)-COALESCE(chase_sr,0) >= 15 THEN 'DOMINANT 1st INNINGS'
        ELSE 'ALL-ROUNDER BATTER'
    END AS batsman_role
FROM pivot
WHERE COALESCE(inn1_runs,0)+COALESCE(chase_runs,0) >= 500
ORDER BY total_runs DESC

Q3 — Bowler Rising Graph
Computes a season-level impact score: wickets×10 − economy×2. Then uses a LAG() window function to compare each season against the previous one. If a bowler improved more than 60% of their season transitions, they're tagged a Rising Star. JJ Bumrah and Mohammed Shami both show 100% improvement rates.

In [0]:
q3 = spark.sql("""
WITH bowler_season AS (
    SELECT
        b.Bowler                                                AS bowler_sk,
        p.Player_Name,
        b.Season                                               AS season_year,
        COUNT(*)                                               AS balls,
        SUM(b.Bowler_Wicket)                                   AS wickets,
        ROUND(SUM(b.Runs_Scored + b.Extra_runs)*6.0/COUNT(*), 2) AS economy,
        ROUND(
            SUM(b.Bowler_Wicket)*10.0
            - SUM(b.Runs_Scored + b.Extra_runs)*6.0/COUNT(*)*2,
        2)                                                     AS season_impact_score
    FROM ball_by_ball b
    JOIN player p ON b.Bowler = p.PLAYER_SK
    GROUP BY b.Bowler, p.Player_Name, b.Season
    HAVING COUNT(*) >= 60
),

trend AS (
    SELECT *,
        LAG(season_impact_score)
            OVER (PARTITION BY bowler_sk ORDER BY season_year) AS prev_impact,
        COUNT(*)
            OVER (PARTITION BY bowler_sk)                      AS total_seasons
    FROM bowler_season
    -- ✅ bowler_sk not Bowler — we're now reading from bowler_season CTE
),

summary AS (
    SELECT
        Player_Name,
        COUNT(*)                                               AS seasons_played,
        SUM(CASE
            WHEN season_impact_score > COALESCE(prev_impact, season_impact_score - 1)
            THEN 1 ELSE 0
        END)                                                   AS seasons_improved,
        SUM(wickets)                                           AS total_wickets,
        ROUND(AVG(season_impact_score), 2)                     AS avg_impact,
        ROUND(
            SUM(CASE
                WHEN season_impact_score > COALESCE(prev_impact, season_impact_score - 1)
                THEN 1 ELSE 0
            END) * 100.0 / NULLIF(COUNT(*) - 1, 0),
        1)                                                     AS improvement_pct
    FROM trend
    WHERE total_seasons >= 3
    GROUP BY Player_Name
)

SELECT *,
    CASE
        WHEN improvement_pct >= 60 THEN 'RISING STAR'
        WHEN improvement_pct >= 40 THEN 'IMPROVING'
        ELSE                            'INCONSISTENT'
    END AS trend_label
FROM summary
WHERE total_wickets >= 30
ORDER BY improvement_pct DESC, avg_impact DESC
""")

q3.show(15, truncate=False)

Q4 — Phase Specialization
Custom milestone score per phase: (wickets/balls)×100×100 + dot%×0.5 − economy×1.5. Threshold of 50 for each phase decides the label. Dual Threats like GC Smith and Shami score above 50 in both powerplay and death overs simultaneously.

In [0]:
%sql
-- Q4: Bowler phase specialization — Powerplay / Middle / Death
-- Phase score = (wkts/balls)*100*100 + dot_pct*0.5 - economy*1.5
WITH phase_stats AS (
    SELECT b.Bowler, p.Player_Name,
        CASE
            WHEN b.Over_id BETWEEN 1  AND 6  THEN 'Powerplay'
            WHEN b.Over_id BETWEEN 7  AND 15 THEN 'Middle'
            WHEN b.Over_id BETWEEN 16 AND 20 THEN 'Death'
        END AS phase,
        COUNT(*)                                                AS balls,
        SUM(b.Bowler_Wicket)                                   AS wickets,
        ROUND(SUM(b.Runs_Scored+b.Extra_runs)*6.0/COUNT(*),2)  AS economy,
        ROUND(SUM(CASE WHEN b.Runs_Scored=0
            AND b.Extra_runs=0 THEN 1 ELSE 0 END)*100.0/COUNT(*),1) AS dot_pct,
        ROUND(SUM(b.Bowler_Wicket)*100.0/NULLIF(COUNT(*),0)*100
              + dot_pct*0.5
              - economy*1.5, 2)                                AS phase_score
    From ball_by_ball b JOIN player p ON b.Bowler=p.PLAYER_SK
    WHERE b.Over_id BETWEEN 1 AND 20
    GROUP BY b.Bowler, p.Player_Name, phase
    HAVING COUNT(*) >= 60
    )
SELECT Player_Name,
    MAX(CASE WHEN phase='Powerplay' THEN phase_score END) AS pp_score,
    MAX(CASE WHEN phase='Death'     THEN phase_score END) AS death_score,
    CASE
        WHEN pp_score>=50 AND death_score>=50 THEN 'DUAL THREAT (PP + Death)'
        WHEN pp_score>=50                     THEN 'POWERPLAY SPECIALIST'
        WHEN death_score>=50                  THEN 'DEATH OVER SPECIALIST'
        ELSE                                       'MIDDLE OVERS SPECIALIST'
    END AS bowler_type
FROM phase_stats
GROUP BY Player_Name
ORDER BY COALESCE(pp_score,0)+COALESCE(death_score,0) DESC

In [0]:
%sql
-- find the matches where score is level
WITH totals AS (
    SELECT 
        match_id,
        Innings_No,
        SUM(Runs_Scored) AS TOTAL
    FROM ball_by_ball
    GROUP BY match_id, Innings_No
)
SELECT 
    match_id,
    Innings_No,
    TOTAL,
    CASE 
        WHEN MAX(TOTAL) OVER (PARTITION BY match_id) 
             = MIN(TOTAL) OVER (PARTITION BY match_id)
        THEN 'Tie'
        ELSE 'Result'
    END AS outcome
FROM totals
ORDER BY match_id, Innings_No;

